# 00 · Getting started — set up FaultyCat & prepare the lab

Gets you from a plugged-in board to a verified, ready-to-use session:
**detect → connect → verify communication → understand the channels →
wiring & safety**. Run it once when you sit down at the bench.

> Assumes the package is already installed (see the repo README). These
> notebooks contain real usage only.

> ⚠️ **Safety.** FaultyCat injects faults. EMFI fires high voltage through
> the coil; crowbar shorts the target's power line. **Keep the plastic
> shield installed** and never touch the exposed HV circuitry. This
> notebook only *reads* status — it never fires.

In [ ]:
import faultycat as fc
print('faultycat', fc.__version__)

## 1 · Detect the board

FaultyCat enumerates as a USB composite device (VID `0x1209` / PID
`0xFA17`) exposing **four** CDC serial interfaces. If nothing shows up:
check the USB cable is data-capable and the board is powered.

In [ ]:
from serial.tools import list_ports
hits = [p for p in list_ports.comports() if p.vid == 0x1209 and p.pid == 0xFA17]
if hits:
    print(f'FaultyCat detected — {len(hits)} CDC interfaces:')
    for p in sorted(hits, key=lambda p: p.device):
        print(f'  {p.device}  {p.interface or ""}')
else:
    print('No board found. You can still explore with the simulator (next cell).')

## 2 · Connect

`fc.connect()` auto-discovers each CDC by VID:PID and opens EMFI, crowbar
and the scanner. **No board handy?** Set `SIM = True` to drive an in-memory
FaultyCat that speaks the real protocols — every notebook here runs on it.

In [ ]:
SIM = False                      # True = no hardware, in-memory simulator
cat = fc.connect(simulator=SIM)
cat                              # renders which engines came up

## 3 · Verify communication

Read each engine's status straight off the hardware. `state=IDLE, err=NONE`
on both means the board is healthy and idle. A clean read here is your
smoke test that host↔firmware framing works.

In [ ]:
print('EMFI   :', dict(cat.emfi.status.as_rows()))
print('CROWBAR:', dict(cat.crowbar.status.as_rows()))
print('SCANNER:', repr(cat.scanner))

## 4 · The four channels

| Interface | Engine | What you drive it with |
| --- | --- | --- |
| CDC0 | **EMFI** | `cat.emfi` — electromagnetic pulse |
| CDC1 | **Crowbar** | `cat.crowbar` — voltage glitch |
| CDC2 | **Scanner shell** | `cat.scanner` (SWD/I2C/logic) + `cat.uart` control |
| CDC3 | **Target UART** | `cat.uart` data — read the target's response |

A campaign (parameter sweep) rides the EMFI or crowbar channel:
`cat.campaign('emfi')` / `cat.campaign('crowbar')`.

## 5 · Wiring for the next notebooks

Depending on what you'll do, wire the target to FaultyCat:

- **Glitching (crowbar)** — target **VCC** → crowbar output (LP or HP), common **GND**.
- **Fault injection (EMFI)** — position the coil over the target die; no electrical contact needed.
- **Trigger (both)** — a target **GPIO that rises just before the code to attack** → FaultyCat **trigger input** (enables reproducible, aligned glitches via `ext_rising`).
- **SWD detection** — target **SWCLK / SWDIO / GND** → any GP channels of the **scanner header**.

Next: `01-glitching` (crowbar), `02-fault-injection` (EMFI), `03-jtagulator-swd` (pinout).

In [ ]:
cat.close()
print('Session closed. Lab ready.')